# GLiNER2 Production NER Pipeline — Geopolitical News (PERSON, ORG, GPE, EVENT, DATE, TIME, QUANTITY)

This notebook replaces the previous specialist / pseudo-labeling approach with a
**single combined dataset, single fine-tuning run** pipeline, and fixes every issue
raised in the review of the earlier notebook.

## Label ownership (final)

| Label | Source | Notes |
|---|---|---|
| `PERSON` | TNER OntoNotes5 | |
| `ORG` | TNER OntoNotes5 | |
| `DATE` | TNER OntoNotes5 | |
| `TIME` | TNER OntoNotes5 | |
| `QUANTITY` | TNER OntoNotes5 | |
| `GPE` | `combined_output.jsonl` **only** | Removed from OntoNotes on purpose (per your instruction) |
| `EVENT` | `combined_output.jsonl` **only** | |

No label is pseudo-labeled, cross-annotated, or predicted by a "teacher" model.
Each source contributes only the label types it is trusted for. Sentences are never
discarded just because they also contain an untrusted label — the untrusted spans
are simply not attached to that record.

## Fixes applied (mapped to the original review)

| # | Severity | Issue | Fix in this notebook |
|---|---|---|---|
| 1 | 🔴 Critical | `combined_records` undefined, crashes on `random.shuffle` | Single explicit `combined_records = onto_records + dataset2_records`, defined right before use, asserted non-empty |
| 2 | 🔴 Critical | Pooled macro F1 hides minority-class failure | Per-label precision/recall/F1 table (`evaluate_dataset`) + micro/macro summary, every label reported separately |
| 3 | 🔴 Critical | Arbitrary pseudo-label threshold (0.85) | **Removed.** No pseudo-labeling / teacher models at all — each source only supplies its trusted labels |
| 4 | 🟠 Major | Duplicate sentences within a source leak across train/test | `deduplicate_records()` hashes normalized text, applied to each source **before** combining, and again verified after the split |
| 5 | 🟠 Major | Non-stratified split can starve rare labels from train/val/test | Composite stratification key built from rare-label presence (`EVENT`, `QUANTITY`, `TIME`, `GPE`), with automatic fallback to plain shuffle if a stratum is too small for `sklearn` |
| 6 | 🟠 Major | Label imbalance ignored (`MAX_PER_LABEL=None`) | `cap_records_by_label()` kept, documented, and label distribution is printed **before you decide** whether to cap |
| 7 | 🟠 Major | Validation set polluted by pseudo-labeling before split | N/A — no pseudo-labeling exists anymore, but the split still happens **before** any training, and val/test are never touched again |
| 8 | 🟠 Major | Unverified `batch_extract_entities` output shape assumption | N/A — no teacher/pseudo-labeling calls in this notebook |
| 9 | 🟡 Minor | Char-to-token alignment assumption | N/A — both sources are already token-indexed (`tokens`/`tags` and `tokenized_text`/`ner`), so no char-span mapping is ever performed |
| 10 | 🟡 Minor | No early stopping | `TrainingConfig(early_stopping=True, early_stopping_patience=2, eval_strategy="epoch", save_best=True)` |
| 11 | 🟡 Minor | EVENT only ever pseudo-labeled | N/A — EVENT is a **directly trusted** label from `combined_output.jsonl`, never synthetic |
| 12 | 🟡 Minor | Evaluation capped at 500 examples | `EVAL_MAX_EXAMPLES = None` by default — evaluates on the **full** test split |

**Additional bugs found and fixed while verifying against the installed `gliner2` package (`pip show gliner2`) that were not in the original review, because the corresponding cells had never actually been executed:**

- `GLiNER2Trainer.train()` does **not** accept `val_data=`. The real keyword is `eval_data=`. The old final-training cell would have crashed with `TypeError` the moment it ran.
- `TrainingDataset.validate()` does **not** accept `strict=`. The real signature is `validate(raise_on_error: bool = True)`. The old validation cell would also have crashed.
- Both are fixed below and were confirmed against the installed package source, not guessed.

## What changed vs. "train 2-3 times"

The old approach trained an OntoNotes specialist, a Dataset-2 specialist, then a pseudo-labeling
pass, then a *third* "final" model. Since the two sources are disjoint in text and each is only
trusted for a fixed subset of labels, none of that is necessary — GLiNER2 (like GLiNER) is trained
with per-example entity supervision, so a single merged dataset where each example only lists the
labels its source is trusted for is sufficient. **This notebook fine-tunes exactly once.**


In [1]:
# 0. Install dependencies (Colab)
!pip -q install -U "gliner2[local]" datasets huggingface_hub scikit-learn pandas tqdm


In [2]:
# 0b. Optional: only needed if you want to persist checkpoints across Colab sessions.
# Skip this cell if you don't want a Drive dependency.
RUN_ON_COLAB = True

if RUN_ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print("Not running on Colab, or Drive mount failed/declined:", e)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration\n\nEverything that controls dataset composition, splitting, and training lives here.

In [3]:
from pathlib import Path
import json
import random
import hashlib
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

# Upload combined_output.jsonl to Colab (left panel -> Files -> upload), or point
# this at a Google Drive path if you mounted Drive above.
DATASET2_PATH = Path("/content/combined_output.jsonl")

WORK_DIR = Path("/content/gliner2_ner")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Final unified ontology
FINAL_LABELS = ["PERSON", "ORG", "GPE", "EVENT", "DATE", "TIME", "QUANTITY"]

# --- Source authority --------------------------------------------------
# GPE is deliberately REMOVED from OntoNotes. It is only trusted from
# combined_output.jsonl. This is the one behavioural change you asked for
# relative to the earlier "GPE from both sources" notebook.
ONTONOTES_KEEP = {"PERSON", "ORG", "DATE", "TIME", "QUANTITY"}
DATASET2_KEEP = { "GPE", "EVENT","PERSON"}

assert ONTONOTES_KEEP | DATASET2_KEEP == set(FINAL_LABELS)
assert "GPE" not in ONTONOTES_KEEP, "GPE must not be sourced from OntoNotes"

# Optional caps. Leave as None until you've looked at the label distribution
# printed below. Only then decide whether dominant labels need capping.
MAX_PER_LABEL = None          # e.g. 12000 to cap PERSON/ORG dominance
MAX_ONTONOTES_RECORDS = None  # e.g. 60000 to bound OntoNotes size
MAX_DATASET2_RECORDS = None

# Fraction of "no relevant entity" sentences from OntoNotes to keep as hard
# negatives, so the model also learns when to predict nothing. 0.0 disables.
NEGATIVE_SAMPLE_RATIO = 0.05

# Evaluation: do NOT cap this for a real production run (fixes issue #12).
EVAL_MAX_EXAMPLES = None

# GLiNER2 base checkpoint
BASE_MODEL = "fastino/gliner2-base-v1"

print("Work dir:", WORK_DIR)
print("Final labels:", FINAL_LABELS)
print("OntoNotes trusted labels:", sorted(ONTONOTES_KEEP))
print("Dataset2 trusted labels:", sorted(DATASET2_KEEP))


Work dir: /content/gliner2_ner
Final labels: ['PERSON', 'ORG', 'GPE', 'EVENT', 'DATE', 'TIME', 'QUANTITY']
OntoNotes trusted labels: ['DATE', 'ORG', 'PERSON', 'QUANTITY', 'TIME']
Dataset2 trusted labels: ['EVENT', 'GPE', 'PERSON']


## 2. Load OntoNotes5

The plain `load_dataset("tner/ontonotes5")` call fails on recent `datasets` versions
because the repo still ships a legacy loading script, so we pull the official
Parquet export directly and resolve the label mapping from `dataset/label.json`
rather than hard-coding numeric tag IDs.

In [4]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download

DATASET_NAME = "tner/ontonotes5"

PARQUET_BASE = (
    "https://huggingface.co/datasets/"
    "tner/ontonotes5/resolve/"
    "refs%2Fconvert%2Fparquet/ontonotes5"
)
TRAIN_URL = f"{PARQUET_BASE}/train/0000.parquet"
VALIDATION_URL = f"{PARQUET_BASE}/validation/0000.parquet"
TEST_URL = f"{PARQUET_BASE}/test/0000.parquet"

ontonotes = load_dataset(
    "parquet",
    data_files={"train": TRAIN_URL, "validation": VALIDATION_URL, "test": TEST_URL},
)

required_columns = {"tokens", "tags"}
missing = required_columns - set(ontonotes["train"].column_names)
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

label_path = hf_hub_download(repo_id=DATASET_NAME, filename="dataset/label.json", repo_type="dataset")
with open(label_path, "r", encoding="utf-8") as f:
    TAG2ID = json.load(f)
ID2TAG = {int(v): str(k) for k, v in TAG2ID.items()}

if not ID2TAG:
    raise RuntimeError("OntoNotes5 label mapping is empty.")

print("Splits:", {k: len(v) for k, v in ontonotes.items()})
print("Num tags:", len(ID2TAG))


Splits: {'train': 59924, 'validation': 8528, 'test': 8262}
Num tags: 37


## 3. Convert OntoNotes5 BIO tags to entity spans (PERSON/ORG/DATE/TIME/QUANTITY only)

`bio_to_entities` decodes tag *names* (not raw numeric IDs) so every `B-`/`I-` variant of the
same family collapses to one canonical label, and anything outside `ONTONOTES_KEEP` — including
`GPE` — is dropped. This is issue #9 ("tokenization alignment") made moot: OntoNotes is already
token-indexed, so entity spans are produced as `[start_token, end_token_inclusive, label]` with
no character-offset math anywhere.

In [5]:
def normalize_ontonotes_label(tag_name):
    tag_name = str(tag_name).strip().upper()
    raw_label = tag_name.split("-", 1)[1] if "-" in tag_name else tag_name
    aliases = {"PER": "PERSON", "PERSON": "PERSON", "ORG": "ORG",
               "DATE": "DATE", "TIME": "TIME", "QUANTITY": "QUANTITY", "GPE": "GPE"}
    return aliases.get(raw_label, raw_label)


def bio_to_entities(tokens, tag_ids, id2tag):
    """Token-index BIO/BIOES decoder -> list of (start, end_exclusive, label)."""
    entities = []
    current = None

    def close_current(end_idx):
        nonlocal current
        if current is not None:
            start, label = current
            entities.append((start, end_idx, label))
            current = None

    for i, tag_id in enumerate(tag_ids):
        tag = id2tag[int(tag_id)]
        if tag == "O":
            close_current(i)
            continue

        prefix, raw_label = (tag.split("-", 1) if "-" in tag else ("B", tag))
        label = normalize_ontonotes_label(raw_label)

        if label not in ONTONOTES_KEEP:
            # Includes GPE, and anything else outside our trusted set.
            close_current(i)
            continue

        if prefix in {"B", "S"}:
            close_current(i)
            if prefix == "S":
                entities.append((i, i + 1, label))
            else:
                current = (i, label)
        elif prefix in {"I", "M"}:
            if current is None or current[1] != label:
                close_current(i)
                current = (i, label)
        elif prefix == "E":
            if current is None or current[1] != label:
                close_current(i)
                entities.append((i, i + 1, label))
            else:
                close_current(i + 1)
        else:
            close_current(i)

    close_current(len(tag_ids))
    return entities


def convert_ontonotes_row(row):
    tokens = row["tokens"]
    spans = bio_to_entities(tokens, row["tags"], ID2TAG)
    entities = [[s, e - 1, label] for s, e, label in spans if tokens[s:e]]
    return {"tokenized_text": tokens, "ner": entities, "source": "ontonotes5"}


sample = convert_ontonotes_row(ontonotes["train"][0])
print(json.dumps(sample, indent=2))
assert all(label != "GPE" for _, _, label in sample["ner"]), "GPE must never appear from OntoNotes"


{
  "tokenized_text": [
    "People",
    "start",
    "their",
    "own",
    "businesses",
    "for",
    "many",
    "reasons",
    "."
  ],
  "ner": [],
  "source": "ontonotes5"
}


In [6]:
# Build the full OntoNotes record list (all splits pooled — we do our OWN
# train/val/test split later, after deduplication, so pooling here is safe
# and does not by itself cause leakage).
random.seed(SEED)

onto_all = []
for split_name, split in ontonotes.items():
    for row in split:
        rec = convert_ontonotes_row(row)
        rec["source_split"] = split_name
        onto_all.append(rec)

onto_with_entities = [r for r in onto_all if r["ner"]]
onto_without_entities = [r for r in onto_all if not r["ner"]]

print("OntoNotes rows with a trusted entity:", len(onto_with_entities))
print("OntoNotes rows with NO trusted entity:", len(onto_without_entities))

# Keep a small slice of true negatives (issue: model never learns "nothing here").
n_negatives = int(len(onto_with_entities) * NEGATIVE_SAMPLE_RATIO)
rng = random.Random(SEED)
rng.shuffle(onto_without_entities)
onto_records = onto_with_entities + onto_without_entities[:n_negatives]

if MAX_ONTONOTES_RECORDS is not None:
    rng.shuffle(onto_records)
    onto_records = onto_records[:MAX_ONTONOTES_RECORDS]

print("OntoNotes records kept (entities + negatives):", len(onto_records))


OntoNotes rows with a trusted entity: 30051
OntoNotes rows with NO trusted entity: 46663
OntoNotes records kept (entities + negatives): 31553


## 4. Load `combined_output.jsonl` (GPE + EVENT only)

Dataset 2 is already token-indexed with inclusive `[start, end, label]` spans, matching the
format `{"tokenized_text": [...], "ner": [[start, end, label], ...]}`. We keep the sentence even
when it contains other label types (e.g. `PERSON`/`ORG` in the example you gave) — we just don't
attach those spans, since `PERSON`/`ORG` are owned by OntoNotes only.

In [7]:
if not DATASET2_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET2_PATH} not found. Upload combined_output.jsonl to Colab "
        "(or update DATASET2_PATH)."
    )

dataset2_records = []
n_bad_lines = 0
n_bad_spans = 0

with DATASET2_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Skipping malformed JSONL line {line_no}: {e}")
            n_bad_lines += 1
            continue

        tokens = obj.get("tokenized_text")
        ner = obj.get("ner", [])
        if not isinstance(tokens, list) or not isinstance(ner, list) or not tokens:
            continue

        filtered = []
        for ann in ner:
            if len(ann) != 3:
                continue
            start, end, label = ann
            label = str(label).upper()
            if label not in DATASET2_KEEP:
                continue
            try:
                start, end = int(start), int(end)
            except (TypeError, ValueError):
                n_bad_spans += 1
                continue
            if start < 0 or end < start or end >= len(tokens):
                n_bad_spans += 1
                continue
            filtered.append([start, end, label])

        dataset2_records.append({
            "tokenized_text": tokens,
            "ner": filtered,
            "source": "dataset2",
        })

if MAX_DATASET2_RECORDS is not None:
    random.Random(SEED).shuffle(dataset2_records)
    dataset2_records = dataset2_records[:MAX_DATASET2_RECORDS]

print("Dataset2 records parsed:", len(dataset2_records))
print("Malformed lines skipped:", n_bad_lines)
print("Malformed spans dropped:", n_bad_spans)
print("Dataset2 label counts (GPE/EVENT only):",
      Counter(a[2] for r in dataset2_records for a in r["ner"]))


Dataset2 records parsed: 19877
Malformed lines skipped: 0
Malformed spans dropped: 0
Dataset2 label counts (GPE/EVENT only): Counter({'GPE': 24577, 'PERSON': 3648, 'EVENT': 2288})


## 5. Deduplicate **within** each source (fixes issue #4)

Even though the two sources are disjoint from each other, each source individually can contain
repeated sentences (OntoNotes has repeated newswire boilerplate; scraped news often repeats
lead sentences across articles). If a duplicate lands in both train and test you get inflated,
meaningless F1. We hash normalized text and keep the first occurrence, **before** any split.

In [8]:
def normalize_for_hash(tokens):
    return hashlib.sha1(" ".join(tokens).strip().lower().encode("utf-8")).hexdigest()


def deduplicate_records(records):
    seen = set()
    out = []
    for rec in records:
        if not rec.get("tokenized_text"):
            continue
        key = normalize_for_hash(rec["tokenized_text"])
        if key in seen:
            continue
        seen.add(key)
        out.append(rec)
    return out


onto_records_before = len(onto_records)
d2_records_before = len(dataset2_records)

onto_records = deduplicate_records(onto_records)
dataset2_records = deduplicate_records(dataset2_records)

print(f"OntoNotes : {onto_records_before:,} -> {len(onto_records):,} after in-source dedup")
print(f"Dataset2  : {d2_records_before:,} -> {len(dataset2_records):,} after in-source dedup")


OntoNotes : 31,553 -> 31,031 after in-source dedup
Dataset2  : 19,877 -> 19,875 after in-source dedup


## 6. Combine into a single dataset (fixes issue #1)

`combined_records` is now defined explicitly, once, right before it's used — no more
`NameError` on `random.shuffle(combined_records)`. No pseudo-labeling, no repeated training
passes: this is the **one** dataset the model will be fine-tuned on.

In [9]:
combined_records = onto_records + dataset2_records
assert len(combined_records) > 0, "combined_records is empty — check upstream cells"

random.Random(SEED).shuffle(combined_records)

# Cross-source safety net: if the same sentence text happens to appear in BOTH
# sources (possible even though the sources are "disjoint" in origin), keep
# only the first occurrence so it can't land in two different splits.
combined_records = deduplicate_records(combined_records)

print("Combined records:", len(combined_records))

def count_entities(records):
    c = Counter()
    for r in records:
        for _, _, label in r["ner"]:
            c[label] += 1
    return c

import pandas as pd

dist_rows = []
for label in FINAL_LABELS:
    dist_rows.append({
        "label": label,
        "from_ontonotes": count_entities(onto_records)[label],
        "from_dataset2": count_entities(dataset2_records)[label],
        "total": count_entities(combined_records)[label],
    })
label_distribution_df = pd.DataFrame(dist_rows)
display(label_distribution_df)


Combined records: 50906


,label,from_ontonotes,from_dataset2,total
0,PERSON,19153,3646,22799
1,ORG,16184,0,16184
2,GPE,0,24576,24576
3,EVENT,0,2288,2288
4,DATE,13935,0,13935
5,TIME,1639,0,1639
6,QUANTITY,860,0,860


## 7. Optional label-aware cap (fixes issue #6)

`MAX_PER_LABEL` stays `None` until you've actually looked at the table above. This function
never blindly undersamples to the smallest class — it ensures every label can contribute up
to the cap without deleting examples that carry a label with no other coverage.

In [10]:
def cap_records_by_label(records, max_per_label, seed=SEED):
    if max_per_label is None:
        return list(records)

    rng = random.Random(seed)
    by_label = defaultdict(list)
    for rec in records:
        for label in {a[2] for a in rec["ner"]}:
            by_label[label].append(rec)

    selected_ids = set()
    for label in FINAL_LABELS:
        candidates = list(by_label.get(label, []))
        rng.shuffle(candidates)
        for rec in candidates[:max_per_label]:
            selected_ids.add(id(rec))

    # Preserve records with no entities (negatives) untouched.
    negatives = [r for r in records if not r["ner"]]
    capped = [r for r in records if id(r) in selected_ids] + negatives
    return capped


combined_records = cap_records_by_label(combined_records, MAX_PER_LABEL, SEED)
print("Combined records after optional cap:", len(combined_records))
print("Label counts after cap:", count_entities(combined_records))


Combined records after optional cap: 50906
Label counts after cap: Counter({'GPE': 24576, 'PERSON': 22799, 'ORG': 16184, 'DATE': 13935, 'EVENT': 2288, 'TIME': 1639, 'QUANTITY': 860})


## 8. Save the combined dataset as JSON\n\nAs requested, the merged dataset is written out as a single JSON array (not JSONL) so it can be inspected, versioned, or reused outside this notebook.

In [11]:
COMBINED_JSON_PATH = WORK_DIR / "combined_dataset.json"

with COMBINED_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(combined_records, f, ensure_ascii=False, indent=2)

print("Saved:", COMBINED_JSON_PATH)
print("Size on disk:", f"{COMBINED_JSON_PATH.stat().st_size / (1024*1024):.2f} MB")
print("Records:", len(combined_records))


Saved: /content/gliner2_ner/combined_dataset.json
Size on disk: 30.12 MB
Records: 50906


## 9. Stratified train / validation / test split (fixes issues #5 and #7)

Splitting happens **after** dedup and **before** anything is done to `train_records`
(no augmentation exists anymore, but the ordering is still enforced so val/test can
never be contaminated by anything derived from the training data).

Because most sentences carry common labels (`PERSON`/`ORG`) while `EVENT`, `QUANTITY`,
`TIME`, and `GPE` are comparatively rare, a plain random split can accidentally starve
one of the splits of a rare label. We stratify on a composite key of which rare labels
are present, with an automatic fallback to a plain shuffle-split if any stratum is too
small for scikit-learn's stratified splitter to handle (minimum 2 members per class).

In [12]:
from sklearn.model_selection import train_test_split

RARE_LABELS_FOR_STRATIFICATION = ["EVENT", "QUANTITY", "TIME", "GPE"]

def stratify_key(record):
    present = {a[2] for a in record["ner"]}
    return tuple(label in present for label in RARE_LABELS_FOR_STRATIFICATION)

strat_keys = [stratify_key(r) for r in combined_records]
key_counts = Counter(strat_keys)
min_stratum_size = min(key_counts.values())

def safe_stratified_split(records, keys, test_size, seed):
    key_counts_local = Counter(keys)
    if min(key_counts_local.values()) < 2:
        print("  -> a stratum has <2 members; falling back to unstratified shuffle-split for this split")
        return train_test_split(records, test_size=test_size, random_state=seed)
    return train_test_split(records, test_size=test_size, random_state=seed, stratify=keys)

print(f"Distinct rare-label strata: {len(key_counts)}, smallest stratum size: {min_stratum_size}")

train_records, temp_records = safe_stratified_split(
    combined_records, strat_keys, test_size=0.20, seed=SEED
)

temp_keys = [stratify_key(r) for r in temp_records]
val_records, test_records = safe_stratified_split(
    temp_records, temp_keys, test_size=0.50, seed=SEED
)

print(f"\ntrain: {len(train_records):,}")
print(f"val:   {len(val_records):,}")
print(f"test:  {len(test_records):,}")

split_dist_rows = []
for name, recs in [("train", train_records), ("val", val_records), ("test", test_records)]:
    counts = count_entities(recs)
    for label in FINAL_LABELS:
        split_dist_rows.append({"split": name, "label": label, "count": counts[label]})
split_distribution_df = pd.DataFrame(split_dist_rows).pivot(index="label", columns="split", values="count")
display(split_distribution_df.reindex(FINAL_LABELS))


Distinct rare-label strata: 7, smallest stratum size: 34

train: 40,724
val:   5,091
test:  5,091


split,test,train,val
label,,,
PERSON,2245,18245,2309
ORG,1610,13015,1559
GPE,2438,19625,2513
EVENT,234,1822,232
DATE,1409,11143,1383
TIME,168,1311,160
QUANTITY,85,697,78


In [13]:
# Leakage guard, part 1: verify no identical sentence text appears in more
# than one split. This must hold BEFORE any training happens.
train_hashes = {normalize_for_hash(r["tokenized_text"]) for r in train_records}
val_hashes = {normalize_for_hash(r["tokenized_text"]) for r in val_records}
test_hashes = {normalize_for_hash(r["tokenized_text"]) for r in test_records}

overlap_train_val = train_hashes & val_hashes
overlap_train_test = train_hashes & test_hashes
overlap_val_test = val_hashes & test_hashes

assert not overlap_train_val, f"Leakage: {len(overlap_train_val)} sentences shared between train/val"
assert not overlap_train_test, f"Leakage: {len(overlap_train_test)} sentences shared between train/test"
assert not overlap_val_test, f"Leakage: {len(overlap_val_test)} sentences shared between val/test"
assert len(train_hashes) + len(val_hashes) + len(test_hashes) == len(train_records) + len(val_records) + len(test_records)

print("No leakage detected across train/val/test.")


No leakage detected across train/val/test.


## 10. Save train/val/test as JSON and as GLiNER2 training JSONL

In [14]:
from gliner2.training.data import InputExample
from collections import defaultdict
import json


def record_to_input_example(record):
    """
    Convert one record into a GLiNER2 InputExample.

    GLiNER2 TrainingDataset does NOT accept examples with zero tasks.
    Therefore, records with no entities return None.
    """
    tokens = record["tokenized_text"]
    text = " ".join(tokens)

    grouped = defaultdict(list)

    for start, end, label in record["ner"]:
        span = " ".join(tokens[start:end + 1]).strip()

        if span:
            grouped[label].append(span)

    entities = {
        label: list(dict.fromkeys(spans))
        for label, spans in grouped.items()
        if spans
    }

    # Important: GLiNER2 validator rejects empty examples.
    if not entities:
        return None

    return InputExample(
        text=text,
        entities=entities
    )


def save_json(records, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)


def examples_to_jsonl(examples, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            obj = {
                "input": ex.text,
                "output": {
                    "entities": ex.entities
                }
            }
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


# ---------------------------------------------------------
# Save ORIGINAL split records
# These files can still contain no-entity examples.
# ---------------------------------------------------------

save_json(train_records, WORK_DIR / "train.json")
save_json(val_records, WORK_DIR / "validation.json")
save_json(test_records, WORK_DIR / "test.json")


# ---------------------------------------------------------
# Convert only entity-bearing records for GLiNER2
# ---------------------------------------------------------

train_examples = []

for record in train_records:
    example = record_to_input_example(record)
    if example is not None:
        train_examples.append(example)


val_examples = []

for record in val_records:
    example = record_to_input_example(record)
    if example is not None:
        val_examples.append(example)


test_examples = []

for record in test_records:
    example = record_to_input_example(record)
    if example is not None:
        test_examples.append(example)


# ---------------------------------------------------------
# Save GLiNER2 JSONL files
# ---------------------------------------------------------

TRAIN_JSONL = WORK_DIR / "train.jsonl"
VAL_JSONL = WORK_DIR / "validation.jsonl"
TEST_JSONL = WORK_DIR / "test.jsonl"

examples_to_jsonl(train_examples, TRAIN_JSONL)
examples_to_jsonl(val_examples, VAL_JSONL)
examples_to_jsonl(test_examples, TEST_JSONL)


# ---------------------------------------------------------
# Statistics
# ---------------------------------------------------------

train_negative_count = sum(
    1 for r in train_records if not r["ner"]
)

val_negative_count = sum(
    1 for r in val_records if not r["ner"]
)

test_negative_count = sum(
    1 for r in test_records if not r["ner"]
)


print("Saved JSON:")
print(" ", WORK_DIR / "train.json")
print(" ", WORK_DIR / "validation.json")
print(" ", WORK_DIR / "test.json")

print("\nSaved GLiNER2 JSONL:")
print(" ", TRAIN_JSONL)
print(" ", VAL_JSONL)
print(" ", TEST_JSONL)

print("\nDataset sizes:")
print(
    f"Train: {len(train_records):,} records "
    f"-> {len(train_examples):,} GLiNER2 examples"
)

print(
    f"Validation: {len(val_records):,} records "
    f"-> {len(val_examples):,} GLiNER2 examples"
)

print(
    f"Test: {len(test_records):,} records "
    f"-> {len(test_examples):,} GLiNER2 examples"
)

print("\nNo-entity records excluded from GLiNER2:")
print(f"Train: {train_negative_count:,}")
print(f"Validation: {val_negative_count:,}")
print(f"Test: {test_negative_count:,}")

Saved JSON:
  /content/gliner2_ner/train.json
  /content/gliner2_ner/validation.json
  /content/gliner2_ner/test.json

Saved GLiNER2 JSONL:
  /content/gliner2_ner/train.jsonl
  /content/gliner2_ner/validation.jsonl
  /content/gliner2_ner/test.jsonl

Dataset sizes:
Train: 40,724 records -> 37,439 GLiNER2 examples
Validation: 5,091 records -> 4,658 GLiNER2 examples
Test: 5,091 records -> 4,647 GLiNER2 examples

No-entity records excluded from GLiNER2:
Train: 3,285
Validation: 433
Test: 444


## 11. Validate the training data with GLiNER2's own validator

**Bug fixed here:** the installed `gliner2` package's `TrainingDataset.validate()` signature is
`validate(self, raise_on_error: bool = True)` — there is no `strict=` kwarg. The earlier notebook
called `validate(strict=True, raise_on_error=True)`, which would have raised a `TypeError` the
moment this cell ran (confirmed against the installed package source, not guessed).

In [15]:
from gliner2.training.data import TrainingDataset

# Safety check: every example must contain at least one entity task.
assert all(
    getattr(example, "entities", None)
    for example in train_examples
), "Empty training example found."

assert all(
    getattr(example, "entities", None)
    for example in val_examples
), "Empty validation example found."


train_dataset = TrainingDataset(train_examples)
val_dataset = TrainingDataset(val_examples)


# Correct GLiNER2 API
train_dataset.validate(raise_on_error=True)
val_dataset.validate(raise_on_error=True)


print("Training dataset stats:")
train_dataset.print_stats()

print("\nValidation dataset stats:")
val_dataset.print_stats()

print("\n✅ Validation passed.")
print("No empty/no-task InputExample remains.")

Training dataset stats:

GLiNER2 Training Dataset Statistics
Total examples: 37439

Text lengths: min=3, max=1398, mean=159.9

Task Distribution:
  entities_only: 37439 (100.0%)

Entity Types (65249 total mentions):
  GPE: 19602
  PERSON: 18044
  ORG: 12698
  DATE: 11087
  EVENT: 1821
  TIME: 1303
  QUANTITY: 694


Validation dataset stats:

GLiNER2 Training Dataset Statistics
Total examples: 4658

Text lengths: min=3, max=1004, mean=159.3

Task Distribution:
  entities_only: 4658 (100.0%)

Entity Types (8164 total mentions):
  GPE: 2511
  PERSON: 2282
  ORG: 1530
  DATE: 1374
  EVENT: 232
  TIME: 159
  QUANTITY: 76


✅ Validation passed.
No empty/no-task InputExample remains.


In [16]:
!pip install --upgrade torchao


## 12. Fine-tune GLiNER2 — once, with early stopping

**Bug fixed here:** `GLiNER2Trainer.train()`'s real signature is
`train(self, train_data=None, eval_data=None)`. The earlier notebook passed `val_data=val_examples`,
which does not exist as a parameter and would have raised a `TypeError` immediately (confirmed
against the installed package source). It is `eval_data=` below.

This is the **only** training run in the whole pipeline — no specialists, no pseudo-labeling pass,
no repeated "final" model.

In [18]:
import torch
from gliner2 import GLiNER2
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

FINAL_MODEL_DIR = WORK_DIR / "final_model"
FINAL_EPOCHS = 8          # early stopping will halt sooner if val loss stops improving
FINAL_BATCH_SIZE = 16

model = GLiNER2.from_pretrained(BASE_MODEL)

config = TrainingConfig(
    output_dir=str(FINAL_MODEL_DIR),
    experiment_name="geopolitical_news_ner_7label",
    num_epochs=FINAL_EPOCHS,
    batch_size=FINAL_BATCH_SIZE,
    gradient_accumulation_steps=2,
    encoder_lr=1e-5,
    task_lr=5e-4,
    warmup_ratio=0.1,
    scheduler_type="cosine",

    # Evaluate + checkpoint every epoch, keep the best, stop early if it stalls.
    eval_strategy="epoch",
    save_best=True,
    metric_for_best="eval_loss",
    greater_is_better=False,
    early_stopping=True,
    early_stopping_patience=2,
    logging_steps=50,

    # LoRA keeps this trainable on a single Colab GPU.
    use_lora=True,
    lora_r=8,
    lora_alpha=16.0,
    lora_dropout=0.0,
    lora_target_modules=["encoder"],
    save_adapter_only=True,

    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = GLiNER2Trainer(model=model, config=config)

train_result = trainer.train(train_data=train_examples, eval_data=val_examples)
print(train_result)
print("Checkpoints written under:", FINAL_MODEL_DIR)


[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first



Validating records: 100%|██████████| 37439/37439 [00:00<00:00, 105648.29record/s]


Training:   0%|          | 0/9352 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/583 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/583 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/583 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/583 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/583 [00:00<?, ?it/s]

{'total_steps': 8190, 'total_epochs': 7, 'total_time_seconds': 5259.1158266067505, 'samples_per_second': 49.812175399267666, 'best_metric': 9.094870268555733, 'train_metrics_history': [{'loss': 123.38544868469238, 'classification_loss': 0.0, 'structure_loss': 80.53170013427734, 'count_loss': 0.0, 'learning_rate': 2.67379679144385e-05, 'epoch': 0.042325780247969215, 'step': 50, 'samples_seen': 1600, 'throughput': 50.86000757009196}, {'loss': 70.51393901824952, 'classification_loss': 0.0, 'structure_loss': 90.82599639892578, 'count_loss': 0.0, 'learning_rate': 5.3475935828877e-05, 'epoch': 0.08507909362975631, 'step': 100, 'samples_seen': 3200, 'throughput': 51.64589046458466}, {'loss': 44.651799840927126, 'classification_loss': 0.0, 'structure_loss': 44.59459686279297, 'count_loss': 0.0, 'learning_rate': 8.021390374331551e-05, 'epoch': 0.1278324070115434, 'step': 150, 'samples_seen': 4800, 'throughput': 52.608324949885564}, {'loss': 36.450439882278445, 'classification_loss': 0.0, 'struc

## 13. Save the model explicitly\n\nBesides whatever the trainer already checkpointed, save the adapter and a full merged copy explicitly so the artifact you hand off doesn't depend on remembering internal checkpoint paths.

In [21]:
from pathlib import Path
import shutil

# ============================================================
# DIRECTORIES
# ============================================================

FINAL_MODEL_DIR = WORK_DIR / "final_model"
FULL_MODEL_DIR = FINAL_MODEL_DIR / "full"

FULL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SAVE FULL MODEL
# ============================================================

model.save_pretrained(str(FULL_MODEL_DIR))

print("Full model saved to:")
print(FULL_MODEL_DIR)

# ============================================================
# VERIFY
# ============================================================

print("\nSaved files:")

for f in FULL_MODEL_DIR.rglob("*"):
    if f.is_file():
        print("  ", f.relative_to(FULL_MODEL_DIR))

# ============================================================
# BACKUP TO GOOGLE DRIVE
# ============================================================

DRIVE_BACKUP_DIR = Path(
    "/content/drive/MyDrive/gliner2_geopolitical_ner"
)

if Path("/content/drive").exists():

    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    shutil.copytree(
        FULL_MODEL_DIR,
        DRIVE_BACKUP_DIR / "full",
        dirs_exist_ok=True
    )

    # Also preserve the trainer-generated final checkpoint
    TRAINER_FINAL_DIR = FINAL_MODEL_DIR / "final"

    if TRAINER_FINAL_DIR.exists():
        shutil.copytree(
            TRAINER_FINAL_DIR,
            DRIVE_BACKUP_DIR / "final",
            dirs_exist_ok=True
        )

    print("\nGoogle Drive backup completed:")
    print("Full model:", DRIVE_BACKUP_DIR / "full")
    print("Trainer final:", DRIVE_BACKUP_DIR / "final")

else:
    print("\nDrive not mounted — skipping backup.")

Full model saved to:
/content/gliner2_ner/final_model/full

Saved files:
   tokenizer_config.json
   model.safetensors
   tokenizer.json
   config.json
   encoder_config/config.json

Google Drive backup completed:
Full model: /content/drive/MyDrive/gliner2_geopolitical_ner/full
Trainer final: /content/drive/MyDrive/gliner2_geopolitical_ner/final


## 14. Reload the saved model from disk (integrity check)

This loads a **brand-new** `GLiNER2` object and attaches the adapter purely from what was written
to disk in the previous cell. If this cell's predictions look sane, the save/load round-trip is
confirmed to work — which is what "properly saved" actually means in practice, not just that
`save_*` ran without throwing.

In [25]:
from gliner2 import GLiNER2
import json
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

ADAPTER_DIR = WORK_DIR / "final_model" / "final"

print("Adapter directory:")
print(ADAPTER_DIR)

# ============================================================
# VERIFY SAVED ADAPTER
# ============================================================

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Adapter directory does not exist:\n{ADAPTER_DIR}"
    )

print("\nAdapter files:")
for f in ADAPTER_DIR.rglob("*"):
    if f.is_file():
        print("  ", f.relative_to(ADAPTER_DIR))

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"\nadapter_config.json not found in:\n{ADAPTER_DIR}"
    )

# ============================================================
# LOAD BASE MODEL
# ============================================================

inference_model = GLiNER2.from_pretrained(BASE_MODEL)

# ============================================================
# LOAD TRAINED ADAPTER
# ============================================================

inference_model.load_adapter(str(ADAPTER_DIR))
inference_model.eval()

print("\n✅ Adapter loaded successfully.")

# ============================================================
# ENTITY TYPES
# ============================================================

ENTITY_TYPES = {
    "PERSON": "Names of individual people.",
    "ORG": "Organizations, companies, institutions, agencies, or groups.",
    "GPE": "Geopolitical entities such as countries, states, provinces, and cities.",
    "EVENT": "Named or identifiable real-world events, conflicts, operations, disasters, or incidents.",
    "DATE": "Calendar dates and date expressions.",
    "TIME": "Clock times and time-of-day expressions.",
    "QUANTITY": "Measured quantities, amounts, counts, or numeric quantities with units.",
}

# ============================================================
# SMOKE TEST TEXT
# ============================================================

smoke_test_text = """
A US- and UK-backed coup d'état in 1953 deposed the democratically-elected
Iranian prime minister Mohammad Mosaddegh due to his nationalization of the
oil industry, strengthening the rule of Shah Mohammad Reza Pahlavi.

Israel maintained ties with Iran as part of its alliance of the periphery
strategy. Resentment of the Shah's autocratic rule led to the 1979 revolution
in which Pahlavi was overthrown and replaced by an Islamic republic.

Iran severed diplomatic ties with the US and Israel and held the American
embassy staff hostage, releasing them after signing the Algiers Accords (1981).

During the Iran–Iraq War, the US supported Iraq. In 1988, a US warship struck
an Iranian mine, and the US responded by attacking Iran's navy.

Iran started a ballistic missile program to deter Iraqi missile attacks on
Iranian cities during the 1980–88 Iran–Iraq War.

In 2005, the US began imposing sanctions targeting Iran's nuclear program,
and in 2006 the United Nations Security Council (UNSC) imposed a series of
sanctions against Iran.

In January 2020, US president Donald Trump ordered the assassination of
Qasem Soleimani, the commander of the Iranian Quds Force.

In June 2025, Israel launched the Twelve-Day War by attacking Iranian military
and nuclear facilities, provoking Iranian counter-strikes.

The United States also joined in support by striking Iranian nuclear
facilities during the Twelve-Day War, which ended in a ceasefire.

In early 2026, Israeli prime minister Benjamin Netanyahu lobbied President
Donald Trump for a joint military strike on Iran, specifically targeting its
leadership. Following high-level meetings in February, Trump authorized
Operation Epic Fury.
"""

# ============================================================
# RUN INFERENCE
# ============================================================

smoke_result = inference_model.extract_entities(
    smoke_test_text,
    ENTITY_TYPES,
    include_confidence=True,
    include_spans=True
)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("SMOKE TEST RESULTS")
print("=" * 70)

print(json.dumps(
    smoke_result,
    indent=2,
    ensure_ascii=False
))

Adapter directory:
/content/gliner2_ner/final_model/final

Adapter files:
   README.md
   adapter_config.json
   adapter_model.safetensors


[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first

✅ Adapter loaded successfully.

SMOKE TEST RESULTS
{
  "entities": {
    "PERSON": [
      {
        "text": "Mohammad Mosaddegh",
        "confidence": 0.9890246987342834,
        "start": 99,
        "end": 117
      },
      {
        "text": "Mohammad Reza Pahlavi",
        "confidence": 0.9839725494384766,
        "start": 197,
        "end": 218
      },
      {
        "text": "Benjamin Netanyahu",
        "confidence": 0.980694055557251,
        "start": 1485,
        "end": 1503
      },
      {
        "text": "Donald Trump",
        "confidence": 0.9709007740020752,
        "start": 1522,
        "end": 1534
      },
      {
        "text": "Qasem Soleimani",
        "confidence": 0.938117265701294,
        "start": 1113,
        "end": 1128
      },
      {
        "text": "Pahlavi",
        "confidence": 0.7438808083534241,
        "start": 38

In [30]:
from pathlib import Path
import shutil
from google.colab import files

FINAL_MODEL_DIR = WORK_DIR / "final_model"

if not FINAL_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Training output not found: {FINAL_MODEL_DIR}"
    )

ZIP_BASE = WORK_DIR / "gliner2_geopolitical_ner_complete"

ZIP_PATH = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=str(FINAL_MODEL_DIR.parent),
    base_dir=FINAL_MODEL_DIR.name
)

print("Created:", ZIP_PATH)
print("Size:", round(Path(ZIP_PATH).stat().st_size / (1024**2), 2), "MB")

files.download(ZIP_PATH)

Created: /content/gliner2_ner/gliner2_geopolitical_ner_complete.zip
Size: 762.21 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
from pathlib import Path
import shutil

SOURCE = Path("/content/gliner2_ner/final_model/final")
DEST = Path("/content/drive/MyDrive/gliner2_geopolitical_ner_final")

shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)

print("✅ Saved permanently to Google Drive:")
print(DEST)

✅ Saved permanently to Google Drive:
/content/drive/MyDrive/gliner2_geopolitical_ner_final


In [33]:
from pathlib import Path

DEST = Path("/content/drive/MyDrive/gliner2_geopolitical_ner_final")

print("Exists:", DEST.exists())

for f in DEST.rglob("*"):
    if f.is_file():
        print(f.relative_to(DEST))

Exists: True
README.md
adapter_config.json
adapter_model.safetensors


In [34]:
from pathlib import Path
import shutil

SOURCE = Path("/content/gliner2_ner/final_model")
DEST = Path("/content/drive/MyDrive/gliner2_geopolitical_ner_complete")

if not SOURCE.exists():
    raise FileNotFoundError(f"Training directory not found: {SOURCE}")

print("Copying complete training directory to Google Drive...")
print("FROM:", SOURCE)
print("TO:  ", DEST)

shutil.copytree(
    SOURCE,
    DEST,
    dirs_exist_ok=True
)

print("\n✅ COMPLETE TRAINING DIRECTORY SAVED TO DRIVE")
print(DEST)

Copying complete training directory to Google Drive...
FROM: /content/gliner2_ner/final_model
TO:   /content/drive/MyDrive/gliner2_geopolitical_ner_complete

✅ COMPLETE TRAINING DIRECTORY SAVED TO DRIVE
/content/drive/MyDrive/gliner2_geopolitical_ner_complete


In [35]:
from pathlib import Path

DEST = Path("/content/drive/MyDrive/gliner2_geopolitical_ner_complete")

print("Exists:", DEST.exists())

for item in sorted(DEST.iterdir()):
    if item.is_dir():
        print(f"📁 {item.name}")
    else:
        print(f"📄 {item.name}")

Exists: True
📁 adapter
📁 best
📁 checkpoint-epoch-4
📁 checkpoint-epoch-5
📁 checkpoint-epoch-6
📁 final
📁 full
📁 logs
📄 training_config.json


## 15. Per-label evaluation on the **full** test set (fixes issues #2 and #12)

Exact-match evaluation on normalized `(entity text, label)` pairs, reported **per label**
plus micro/macro summaries — a model that nails `PERSON`/`ORG` and completely fails `EVENT`
will now show a low `EVENT` row instead of hiding behind a good pooled score. No `max_examples`
cap by default (`EVAL_MAX_EXAMPLES = None`), so this runs over the entire held-out test split.

In [32]:
import re

def normalize_text(s):
    return re.sub(r"\s+", " ", s.strip().lower())

def extract_gold(record):
    gold = set()
    for start, end, label in record["ner"]:
        span = " ".join(record["tokenized_text"][start:end + 1])
        gold.add((normalize_text(span), label))
    return gold

def extract_pred(model, text, entity_types):
    result = model.extract_entities(text, entity_types, include_confidence=False, include_spans=False)
    pred = set()
    for label, values in result.get("entities", {}).items():
        label = label.upper()
        for value in values:
            if isinstance(value, dict):
                value = value.get("text", "")
            pred.add((normalize_text(str(value)), label))
    return pred

def evaluate_dataset(model, records, entity_types, max_examples=None):
    if max_examples is not None:
        records = records[:max_examples]

    per_label = {label: Counter() for label in FINAL_LABELS}  # tp/fp/fn per label
    for label in FINAL_LABELS:
        per_label[label].update({"tp": 0, "fp": 0, "fn": 0})

    for rec in records:
        text = " ".join(rec["tokenized_text"])
        gold = extract_gold(rec)
        pred = extract_pred(model, text, entity_types)

        for label in FINAL_LABELS:
            gold_l = {g for g in gold if g[1] == label}
            pred_l = {p for p in pred if p[1] == label}
            per_label[label]["tp"] += len(gold_l & pred_l)
            per_label[label]["fp"] += len(pred_l - gold_l)
            per_label[label]["fn"] += len(gold_l - pred_l)

    rows = []
    micro_tp = micro_fp = micro_fn = 0
    for label in FINAL_LABELS:
        tp, fp, fn = per_label[label]["tp"], per_label[label]["fp"], per_label[label]["fn"]
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({"label": label, "precision": round(precision, 4), "recall": round(recall, 4),
                      "f1": round(f1, 4), "support": tp + fn, "tp": tp, "fp": fp, "fn": fn})
        micro_tp += tp
        micro_fp += fp
        micro_fn += fn

    per_label_df = pd.DataFrame(rows)

    micro_p = micro_tp / (micro_tp + micro_fp) if (micro_tp + micro_fp) else 0.0
    micro_r = micro_tp / (micro_tp + micro_fn) if (micro_tp + micro_fn) else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) else 0.0
    macro_f1 = per_label_df["f1"].mean()

    summary = {
        "n_examples": len(records),
        "micro_precision": round(micro_p, 4),
        "micro_recall": round(micro_r, 4),
        "micro_f1": round(micro_f1, 4),
        "macro_f1": round(float(macro_f1), 4),
    }
    return per_label_df, summary


test_per_label_df, test_summary = evaluate_dataset(
    inference_model, test_records, ENTITY_TYPES, max_examples=EVAL_MAX_EXAMPLES
)

display(test_per_label_df)
print("\nOverall:", json.dumps(test_summary, indent=2))


KeyboardInterrupt: 

## 16. Final leakage audit\n\nSame check as before, re-run explicitly right before you accept these numbers, so the evaluation cell above can never silently be reporting inflated scores.

In [ ]:
assert len(train_hashes & test_hashes) == 0, "Leakage detected: train/test overlap"
assert len(val_hashes & test_hashes) == 0, "Leakage detected: val/test overlap"
assert len(train_hashes & val_hashes) == 0, "Leakage detected: train/val overlap"

total_after_split = len(train_records) + len(val_records) + len(test_records)
assert total_after_split == len(combined_records), (
    f"Split sizes ({total_after_split}) don't match combined_records ({len(combined_records)})"
)

print("Confirmed: no train/val/test text overlap, and split sizes are consistent.")
print(f"Evaluated on {test_summary['n_examples']} of {len(test_records)} test records "
      f"({'full test set' if EVAL_MAX_EXAMPLES is None else 'capped'}).")


## 17. Summary & recommended next steps

- The combined dataset is saved at `combined_dataset.json` (single JSON array, both sources
  merged, GPE sourced only from `combined_output.jsonl`).
- Train/val/test splits are saved as both JSON arrays and GLiNER2-format JSONL.
- The model was fine-tuned **once**, with early stopping on `eval_loss` (patience 2).
- Per-label P/R/F1 on the full test set is in `test_per_label_df` above — check `EVENT`,
  `QUANTITY`, and `TIME` specifically since they have the least support.
- The adapter is saved at `ADAPTER_DIR` and was independently reloaded to confirm the
  save/load round-trip works.

**Before treating this as production-ready:**

1. If `label_distribution_df` (Step 6) shows heavy `PERSON`/`ORG` dominance, set `MAX_PER_LABEL`
   in the config cell and re-run from Step 7 onward.
2. If `EVENT`/`QUANTITY`/`TIME` F1 is materially worse than `PERSON`/`ORG`/`GPE`, that's support
   size, not a bug — collect more `combined_output.jsonl`-style examples for those labels rather
   than reintroducing pseudo-labeling.
3. Replace the exact-match evaluator with a manually spot-checked sample (50-100 test sentences)
   to catch annotation-policy disagreements exact match can't see (partial spans, boundary
   differences like "the United States" vs "United States").
4. Re-run the full notebook end-to-end at least once before shipping, since several cells in the
   previous version of this notebook had never actually been executed and contained parameter
   names that don't exist in the installed `gliner2` package.
